# Exp2 evaluation: aggregated vs temporal (sequence) models

Compares models on the same 15,000 test applicants (RQ2.1), then breaks the results down by applicants' history availability and richness (RQ2.2).

**How to use**
1. Train your models on `split == 'train'` and tune on `split == 'val'` (from `clean_data/splits.csv`).
2. Predict a score for **every val and test applicant** with each model.
3. Put them in `MODELS` below and **Run All**.

If `MODELS` is left empty, the notebook runs a demo with two dummy models so you can see the output format.

In [1]:
import sys
from pathlib import Path
import pandas as pd

HERE = next(p for p in [Path.cwd(), Path.cwd() / "outputs" / "exp2"] if (p / "eval_exp2.py").exists())
sys.path.insert(0, str(HERE))
sys.dont_write_bytecode = True
import eval_exp2 as ev

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

## 1. Your models

In [2]:
# V6 additive chain: M0 is the baseline (aggregated features only).
# M1 adds handcrafted temporal features, M2 adds LSTM OOF score, M3 adds both.
MODELS = {
    "M0 (Aggregated)": "v6_pred_M0.csv",
    "M1 (+Temporal)":  "v6_pred_M1.csv",
    "M2 (+LSTM)":      "v6_pred_M2.csv",
    "M3 (+Both)":      "v6_pred_M3.csv",
}
N_BOOT = 1000   # bootstrap resamples for the 95% confidence intervals (200 is faster for a quick look)
SAVE = True     # also save the result tables as CSV files in this folder
SEGMENTS = None  # None = default groups (history_segments.parquet), or a CSV path with SK_ID_CURR, segment

In [3]:
DEMO = not MODELS
if DEMO:
    # Two deliberately simple logistic regressions, trained on train only, so you can see what the output looks like.
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    app = ev.attach_split(pd.read_parquet(ev.CLEAN_DIR / "application.parquet",
                                          columns=["SK_ID_CURR", "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3", "DAYS_BIRTH"]))
    for name, cols in [("Demo small", ["EXT_SOURCE_2", "DAYS_BIRTH"]),
                       ("Demo bigger", ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3", "DAYS_BIRTH"])]:
        m = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression())
        m.fit(app.loc[app.split == "train", cols], app.loc[app.split == "train", "TARGET"])
        scored = app[app.split.isin(["val", "test"])]
        MODELS[name] = pd.DataFrame({"SK_ID_CURR": scored.SK_ID_CURR, "score": m.predict_proba(scored[cols])[:, 1]})
    print("MODELS was empty, so this is a DEMO with two dummy models. These numbers are NOT results.")

## 2. Results

In [4]:
table, comps, preds = ev.compare_prediction_files(list(MODELS.values()), labels=list(MODELS), n_boot=N_BOOT)

### Test-set metrics
One row per model. Each cell shows **value (95% confidence interval)**. The threshold used for precision, recall and
F1 was chosen on the **validation** set (the value that maximises F1), so the test set stays untouched.

| Metric | Meaning |
|---|---|
| `roc_auc` | how well the model ranks defaulters above non-defaulters (0.5 = random, 1 = perfect) |
| `pr_auc` | precision-recall area; more informative with only ~8% defaulters (8% = random) |
| `precision` / `recall` / `f1` | at the chosen threshold: share of flagged who default / share of defaulters flagged / balance of both |
| `precision@top10%` / `recall@top10%` | if the riskiest 10% of applicants are reviewed: share who default / share of all defaulters caught |

In [5]:
ev.pretty_metrics(table)

,threshold,roc_auc,pr_auc,precision,recall,f1,precision@top10%,recall@top10%
model,,,,,,,,
M0 (Aggregated),0.126,0.759 (0.744-0.772),0.248 (0.226-0.271),0.217 (0.203-0.232),0.500 (0.472-0.528),0.302 (0.285-0.320),0.271 (0.249-0.293),0.335 (0.312-0.360)
M1 (+Temporal),0.157,0.760 (0.745-0.774),0.253 (0.231-0.277),0.248 (0.230-0.266),0.416 (0.389-0.445),0.310 (0.291-0.330),0.273 (0.251-0.296),0.339 (0.315-0.362)
M2 (+LSTM),0.138,0.758 (0.744-0.772),0.245 (0.224-0.269),0.230 (0.215-0.248),0.459 (0.431-0.486),0.307 (0.289-0.326),0.262 (0.241-0.284),0.325 (0.301-0.347)
M3 (+Both),0.147,0.761 (0.746-0.775),0.251 (0.229-0.275),0.239 (0.223-0.257),0.442 (0.413-0.469),0.310 (0.292-0.329),0.263 (0.241-0.285),0.325 (0.302-0.349)


### Is each model better than the baseline?
Same test applicants for every model, so the comparison is paired. **Significantly better** means the 95% confidence
interval of the difference lies entirely above 0. The p-value comes from the DeLong test for ROC-AUC and from the
paired bootstrap for PR-AUC.

In [6]:
ev.pretty_comparisons(comps)

,model,vs_baseline,metric,difference (95% CI),p_value,verdict
0,M1 (+Temporal),M0 (Aggregated),roc_auc,+0.0009 (-0.0022 to +0.0041),0.57,no significant difference
1,M1 (+Temporal),M0 (Aggregated),pr_auc,+0.0049 (-0.0016 to +0.0106),0.15,no significant difference
2,M2 (+LSTM),M0 (Aggregated),roc_auc,-0.0003 (-0.0032 to +0.0025),0.82,no significant difference
3,M2 (+LSTM),M0 (Aggregated),pr_auc,-0.0026 (-0.0084 to +0.0030),0.35,no significant difference
4,M3 (+Both),M0 (Aggregated),roc_auc,+0.0019 (-0.0015 to +0.0054),0.29,no significant difference
5,M3 (+Both),M0 (Aggregated),pr_auc,+0.0031 (-0.0035 to +0.0096),0.35,no significant difference


### Results by history segment (RQ2.2)
Same metrics for each group of applicants, each model at its own validation threshold.
* `HISTORY_SEGMENT`: internal (Home Credit) and/or external (bureau) history, or none.
* `INTERNAL_RICHNESS`: quartile of the amount of internal history (Q1 = least, Q4 = most).

Groups with fewer than 20 defaulters show no metrics (too few to be reliable). To use your own groups, set `SEGMENTS`
to a CSV with columns `SK_ID_CURR, segment`.

In [7]:
seg = ev.load_segments(SEGMENTS)
by_seg = ev.segment_table(preds, seg, ["CUSTOM_SEGMENT"] if SEGMENTS else ["HISTORY_SEGMENT", "INTERNAL_RICHNESS"])
ev.segment_pivot(by_seg, "roc_auc").style.format({"default_rate": "{:.2%}", "n": "{:,.0f}", "defaulters": "{:,.0f}"}, precision=3)

In [8]:
ev.segment_pivot(by_seg, "pr_auc").style.format({"default_rate": "{:.2%}", "n": "{:,.0f}", "defaulters": "{:,.0f}"}, precision=3)

## 3. Save

In [9]:
if SAVE and not DEMO:
    table.to_csv(HERE / "exp2_metrics.csv")              # numeric values incl. CI bounds
    comps.to_csv(HERE / "exp2_comparisons.csv", index=False)
    by_seg.to_csv(HERE / "exp2_by_segment.csv", index=False)
    print("Saved result tables in", HERE)
else:
    print("Not saved (demo mode or SAVE = False).")

Saved result tables in /Users/yichuan/Desktop/NUS/BT4012/BT4012 home credit/outputs/exp2
